In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY ===
N_NORMAL = 2000
N_FRAUD  = 100
# =================

# Zadanie 1.1 — Przygotuj dane

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

# Zadanie 1.2 — Podziel dane, wytrenuj, oceń

features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# 1. Podział 80/20 ze stratyfikacją (zachowuje proporcje klas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y         
)

print(f"\nTrain: {len(X_train)} próbek | Test: {len(X_test)} próbek")
print(f"Fraud w train: {y_train.mean():.1%} | Fraud w test: {y_test.mean():.1%}")

# 2. Trening RandomForest (100 drzew)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Ocena modelu
y_pred = model.predict(X_test)
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

# 4. Zapis modelu do pliku
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("\nModel zapisany do 'fraud_model.pkl'")

# Weryfikacja — wczytaj i zrób predykcję testową
with open('fraud_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

sample = pd.DataFrame([{'amount': 8000, 'is_electronics': 1, 'tx_per_minute': 10}])
print(f"Testowa predykcja (fraud?): {bool(loaded_model.predict(sample)[0])}")

Dataset: 2100 wierszy, fraud rate: 4.8%

Train: 1680 próbek | Test: 420 próbek
Fraud w train: 4.8% | Fraud w test: 4.8%

=== Classification Report ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       400
       Fraud       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420


Model zapisany do 'fraud_model.pkl'
Testowa predykcja (fraud?): True
